# TrustLap: full-size GPU training (SPEC 7.9 budgets)

Runs the whole pipeline on a GPU: track banks → blind driver (PBT 8 × 150M steps) → teacher
population (PBT 16 × 100M) → 5 students (40M each) → the full evaluation (200 test tracks × 3 seeds).
Every stage checkpoints and resumes, so if the session dies, just run all again.

**Kaggle (recommended):** Settings → Accelerator → **GPU T4 ×2** (or P100), Internet **on**. Then
**Save Version → Save & Run All (Commit)**: it runs in the background, no tab needed. Outputs land in
`/kaggle/working/trustlap_full.zip`.
**Colab:** Runtime → Change runtime type → T4 GPU → Run all (keep the tab open; checkpoints go to Drive if mounted).

In [ ]:
import os

import jax

print("jax", jax.__version__, jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU visible: enable a GPU accelerator and rerun."
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
print("work dir:", WORK)

In [ ]:
%cd {WORK}
!rm -rf trustlap && git clone -q --depth 1 https://github.com/hitenjain22/trustlap.git
!pip install -q -e trustlap/sim
!cd trustlap && git log -1 --format='commit %h %s'

In [ ]:
# Persist runs outside the repo so a re-run resumes (Colab: optionally mount Drive first).
RUNS = os.path.join(WORK, "trustlap_runs")
os.makedirs(RUNS, exist_ok=True)
if not os.path.islink("trustlap/sim/runs"):
    !rm -rf trustlap/sim/runs && ln -s {RUNS} trustlap/sim/runs

In [ ]:
# The whole pipeline (several hours on a T4).
# Progress: trustlap_runs/full/pipeline.log and each run's log.jsonl.
!cd trustlap/sim && bash scripts/full_pipeline.sh runs/full 2>&1 | grep -v Warning | tail -n 50

In [ ]:
# Package the results for download (policies, logs, lineage, snapshots, metrics).
ZIP = f"{WORK}/trustlap_full.zip"
!cd {RUNS} && zip -qr {ZIP} full -x '*/checkpoint.msgpack' && ls -lh {ZIP}